# I. Introduction

## Research question

Does incorporating weather information improve the out-of-sample prediction of agricultural commodity returns relative to financial information alone?

This project investigates whether weather-related variables provide incremental predictive information for **corn and soybean futures**. It compares financial-only forecasting models with specifications that additionally incorporate a composite Weather Stress Index (WSI) or its underlying weather components.

## Empirical design

- **Commodities:** corn and soybeans.
- **Forecast target:** next-week futures log return.
- **Frequency:** weekly observations ending on Friday.
- **Sample:** 2000–2025 for prices; weather observations begin in 1990 to construct historical climatologies.
- **Evaluation:** expanding-window, annual walk-forward forecasting, starting in 2010.
- **Models:** historical-mean benchmark, Ridge Regression, Random Forest and Histogram-based Gradient Boosting.
- **Information sets:** financial variables only; financial variables plus the WSI; and financial variables plus individual weather components.

Weather variables are lagged by one week to reduce the risk of using information that was not yet available when the forecast was made.

The Weather Stress Index (WSI) is a project-specific adaptation informed by two studies: *Guo, Ji and Zhang (2024), A dataset to measure global climate physical risk*, which inspired the composite index construction, and *Ma, Zhou and Li (2024), Weathering market swings: Does climate risk matter for agricultural commodity price predictability?*, which motivated the PCA-based alternative explored in this project. Our final WSI adapts these approaches to production-weighted agricultural weather indicators and past-only normalisation; it does not exactly replicate either paper.

The objective is to assess incremental predictive performance, not to establish a causal relationship between weather and commodity returns.

In [1]:
# Core libraries
from pathlib import Path
import warnings
import random

import numpy as np
import pandas as pd
import requests
import xarray as xr
import yfinance as yf

# Visualisation
import matplotlib.pyplot as plt

# Machine learning
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

# Statistical tests
from scipy import stats

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

warnings.filterwarnings("ignore", category=FutureWarning)

# Project configuration
START_DATE = "2000-01-01"
END_DATE = "2025-12-31"
WEATHER_START = "1990-01-01"

WEEK_RULE = "W-FRI"
FIRST_TEST_YEAR = 2010
WEATHER_LAG = 1

COMMODITIES = ["corn", "soybeans"]

MARKET_FEATURES = [
    "ret_1w",
    "ret_4w",
    "ret_12w",
    "vol_4w",
    "vol_12w",
    "month_sin",
    "month_cos",
]

WSI_FEATURES = ["wsi_cpri", "in_season"]

COMPONENT_FEATURES = [
    "heat",
    "warm_nights",
    "heavy_rain",
    "frost",
    "dryness",
    "in_season",
]

FEATURE_SETS = {
    "M1_Financial": MARKET_FEATURES,
    "M2_WSI": MARKET_FEATURES + WSI_FEATURES,
    "M3_Components": MARKET_FEATURES + COMPONENT_FEATURES,
}

# Local working directories (created by the notebook)
ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"

for directory in [DATA_RAW, DATA_PROCESSED, RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:", ROOT)
print("Weekly frequency:", WEEK_RULE)
print("Analysis period:", START_DATE, "to", END_DATE)
print("Weather history starts:", WEATHER_START)
print("First out-of-sample year:", FIRST_TEST_YEAR)
print("Commodities:", COMMODITIES)

Project directory: /Users/quentinfosalva/Documents/Python_Projects/weather-alpha-for-commodities
Weekly frequency: W-FRI
Analysis period: 2000-01-01 to 2025-12-31
Weather history starts: 1990-01-01
First out-of-sample year: 2010
Commodities: ['corn', 'soybeans']


## Data retrieval and reproducibility

All datasets are retrieved directly by the notebook from public sources (Yahoo Finance, NOAA CPC and MapSPAM).

Downloaded and processed data are cached locally to avoid unnecessary repeated downloads. Missing or invalid files are automatically retrieved again.

# II. Data Collection and Preparation

## A. Financial Data

Daily futures prices for corn and soybeans are retrieved from Yahoo Finance using the continuous futures tickers `ZC=F` and `ZS=F`, respectively.

The sample covers 2000–2025. Daily closing prices are converted into log returns and aggregated into Friday-ending weekly observations (`W-FRI`).

To limit artificial price movements caused by futures contract switching, estimated roll-day returns and the preceding expiry-day returns are excluded. Weekly observations are retained only when at least three valid daily returns are available.

The resulting weekly series will be used to construct financial predictors and the next-week return forecasting target.

The required raw datasets are downloaded from their respective public sources using the code cell below, this process may take aproximately 30min depending on download speed and data availability.

In [ ]:
# ============================================================
# DOWNLOAD ALL RAW DATA
# Corn and soybeans: prices, ONI, MapSPAM and CPC weather
# ============================================================

import io
import zipfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import product

# ------------------------------------------------------------
# 1. Directories and project parameters
# ------------------------------------------------------------

for folder in [
    DATA_RAW / "cpc",
    DATA_RAW / "enso",
    DATA_RAW / "spam",
]:
    folder.mkdir(parents=True, exist_ok=True)

START_DATE = "2000-01-01"
END_DATE = "2025-12-31"
WEATHER_START = "1990-01-01"

COMMODITY_TICKERS = {
    "corn": "ZC=F",
    "soybeans": "ZS=F",
}

# Only regions required by corn and soybeans
BOXES = {
    "north_america": (24.0, 50.0, -125.0, -66.0),
    "south_america": (-40.0, 13.0, -80.0, -34.0),
}

CPC_VARIABLES = {
    "tmax": "cpc_global_temp/tmax",
    "tmin": "cpc_global_temp/tmin",
    "precip": "cpc_global_precip/precip",
}


# ------------------------------------------------------------
# 2. Daily futures prices — Yahoo Finance
# ------------------------------------------------------------

print("Downloading daily futures prices...")

closes = {}

for commodity, ticker in COMMODITY_TICKERS.items():

    data = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        progress=False,
        auto_adjust=False,
    )

    if data.empty:
        raise RuntimeError(f"No Yahoo Finance data returned for {ticker}")

    close = data["Close"].squeeze()

    closes[commodity] = close

    print(f"{commodity:10s} {ticker:6s} {len(data)} observations")

prices_daily = pd.DataFrame(closes)
prices_daily.index.name = "date"
prices_daily = prices_daily.sort_index()

prices_daily.to_parquet(DATA_RAW / "prices_daily.parquet")

print("Prices saved:", prices_daily.shape)


# ------------------------------------------------------------
# 3. ONI — NOAA CPC
# ------------------------------------------------------------

print("\nDownloading Oceanic Niño Index...")

ONI_URL = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

response = requests.get(ONI_URL, timeout=60)
response.raise_for_status()

oni_text = response.text

oni_path = DATA_RAW / "enso" / "oni.txt"
oni_path.write_text(oni_text)

SEASONS = [
    "DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ",
    "JJA", "JAS", "ASO", "SON", "OND", "NDJ"
]

CENTRAL_MONTH = {
    season: i + 1
    for i, season in enumerate(SEASONS)
}

table = pd.read_csv(
    io.StringIO(oni_text),
    sep=r"\s+",
)

dates = pd.to_datetime(
    dict(
        year=table["YR"],
        month=table["SEAS"].map(CENTRAL_MONTH),
        day=1,
    )
)

oni = pd.Series(
    table["ANOM"].values,
    index=dates,
    name="oni",
).rename_axis("date")

oni.to_frame().to_parquet(
    DATA_RAW / "enso" / "oni_monthly.parquet"
)

print(
    f"ONI saved: {oni.index.min().date()} "
    f"to {oni.index.max().date()}, {len(oni)} months"
)


# ------------------------------------------------------------
# 4. MapSPAM 2010 — agricultural production
# ------------------------------------------------------------

print("\nPreparing MapSPAM production data...")

SPAM_PATH = DATA_RAW / "spam" / "spam2010_production.parquet"
SPAM_ZIP_PATH = DATA_RAW / "spam" / "spam2010_prod_csv.zip"

if SPAM_PATH.exists():

    spam = pd.read_parquet(SPAM_PATH)
    print(f"Existing MapSPAM data loaded: {len(spam):,} rows")

else:

    SPAM_FILE_ID = 3984975
    SPAM_URL = (
        "https://dataverse.harvard.edu/api/access/datafile/"
        f"{SPAM_FILE_ID}"
    )

    if not SPAM_ZIP_PATH.exists():

        response = requests.get(SPAM_URL, timeout=600)
        response.raise_for_status()

        SPAM_ZIP_PATH.write_bytes(response.content)

    crops = ["maiz", "soyb"]

    with zipfile.ZipFile(SPAM_ZIP_PATH) as archive:

        with archive.open("spam2010V2r0_global_P_TA.csv") as f:

            keep = {
                "iso3": "iso3",
                "name_adm1": "adm1",
                "y": "lat",
                "x": "lon",
                "maiz_a": "maiz",
                "soyb_a": "soyb",
            }

            raw = pd.read_csv(
                f,
                usecols=list(keep),
                encoding="latin-1",
            )

    spam = raw.rename(columns=keep)

    spam["iso3"] = spam["iso3"].str.upper()

    spam = spam[
        spam[crops].sum(axis=1) > 0
    ].reset_index(drop=True)

    spam.to_parquet(SPAM_PATH)

    print(f"MapSPAM saved: {len(spam):,} production cells")


# ------------------------------------------------------------
# 5. CPC weather data — NOAA Global Unified Daily Grids
# ------------------------------------------------------------

print("\nDownloading CPC weather data...")

CPC_URL = "https://downloads.psl.noaa.gov/Datasets/{path}.{year}.nc"

MAX_ATTEMPTS = 4

# NetCDF operations are protected by a lock because the library
# is not thread-safe.
NETCDF_LOCK = threading.Lock()


def box_path(box, variable, year):
    return (
        DATA_RAW
        / "cpc"
        / box
        / f"{variable}_{year}.nc"
    )


def cut_box(ds, box):
    """Extract one geographical box from a global CPC file."""

    lat_min, lat_max, lon_min, lon_max = BOXES[box]

    # Convert longitudes from [0, 360] to [-180, 180]
    ds = ds.assign_coords(
        lon=(ds["lon"] + 180) % 360 - 180
    ).sortby("lon")

    lat_slice = (
        slice(lat_max, lat_min)
        if ds["lat"][0] > ds["lat"][-1]
        else slice(lat_min, lat_max)
    )

    return ds.sel(
        lat=lat_slice,
        lon=slice(lon_min, lon_max),
    )


def valid_cpc_file(path, variable):

    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with NETCDF_LOCK:
            with xr.open_dataset(path) as ds:
                return variable in ds.data_vars

    except Exception:
        return False


def download_one(variable, year):
    """Download one global file and extract the required regions."""

    todo = []

    for box in BOXES:

        path = box_path(box, variable, year)

        if not valid_cpc_file(path, variable):
            todo.append(box)

    if not todo:
        return f"skip {variable}_{year}"

    tmp = (
        DATA_RAW
        / "cpc"
        / f"global_{variable}_{year}.nc.part"
    )

    tmp.parent.mkdir(parents=True, exist_ok=True)

    url = CPC_URL.format(
        path=CPC_VARIABLES[variable],
        year=year,
    )

    # Download with retries
    for attempt in range(1, MAX_ATTEMPTS + 1):

        try:

            with requests.get(
                url,
                stream=True,
                timeout=300,
            ) as response:

                response.raise_for_status()

                with open(tmp, "wb") as f:

                    for chunk in response.iter_content(
                        chunk_size=1 << 20
                    ):
                        if chunk:
                            f.write(chunk)

            break

        except requests.RequestException as error:

            if attempt == MAX_ATTEMPTS:

                tmp.unlink(missing_ok=True)

                return (
                    f"FAIL {variable}_{year}: {error}"
                )

            time.sleep(10 * attempt)

    # Extract regional files under the NetCDF lock
    try:

        with NETCDF_LOCK:

            with xr.open_dataset(tmp) as ds:

                for box in todo:

                    out = box_path(box, variable, year)
                    out.parent.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    extracted = cut_box(
                        ds[[variable]],
                        box,
                    ).load()

                    extracted.to_netcdf(out)

        tmp.unlink(missing_ok=True)

        return (
            f"ok {variable}_{year}: "
            f"{len(todo)} regional files"
        )

    except Exception as error:

        tmp.unlink(missing_ok=True)

        return f"FAIL {variable}_{year}: {error}"


# Download from 1990 through 2025
years = range(
    pd.Timestamp(WEATHER_START).year,
    pd.Timestamp(END_DATE).year + 1,
)

jobs = list(
    product(CPC_VARIABLES, years)
)

print(
    f"Downloading {len(jobs)} global CPC files "
    f"and extracting {len(BOXES)} regions..."
)

with ThreadPoolExecutor(max_workers=8) as pool:

    for status in pool.map(
        lambda job: download_one(*job),
        jobs,
    ):
        print(status, flush=True)


# ------------------------------------------------------------
# 6. Final download summary
# ------------------------------------------------------------

print("\n========== DOWNLOAD SUMMARY ==========")

print("\nPrices:")
print(prices_daily.shape)
print(prices_daily.index.min(), "to", prices_daily.index.max())

print("\nONI:")
print(oni.index.min(), "to", oni.index.max())

print("\nMapSPAM:")
print(spam.shape)

print("\nCPC files:")

for box in BOXES:

    for variable in CPC_VARIABLES:

        files = list(
            (DATA_RAW / "cpc" / box).glob(
                f"{variable}_*.nc"
            )
        )

        print(
            f"{box:16s} {variable:8s}: "
            f"{len(files)} files"
        )

print("\nCell 5 completed.")


corn       ZC=F   6369 observations
soybeans   ZS=F   6361 observations
Prices saved: (6408, 2)

ONI saved: 1950-01-01 to 2026-09-01, 921 months

Preparing MapSPAM production data...
Existing MapSPAM data loaded: 608,865 rows

skip tmax_1990


HDF5-DIAG: Error detected in HDF5 (2.2.0) thread 8:
  #000: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5F.c line 490 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4109 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4045 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5Fint.c line 1084 in H5F__i

ok tmax_1991: 2 regional files
skip tmax_1992
skip tmax_1993
skip tmax_1994
skip tmax_1995
skip tmax_1996
skip tmax_1997
skip tmax_1998


HDF5-DIAG: Error detected in HDF5 (2.2.0) thread 8:
  #000: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5F.c line 490 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4109 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4045 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5Fint.c line 1084 in H5F__i

ok tmax_1999: 2 regional files
ok tmax_2000: 2 regional files
ok tmax_2001: 2 regional files
ok tmax_2002: 2 regional files


HDF5-DIAG: Error detected in HDF5 (2.2.0) thread 2:
  #000: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5F.c line 490 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4109 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4045 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5Fint.c line 1084 in H5F__i

... (114 outputs left out)

ok precip_2024: 2 regional files


HDF5-DIAG: Error detected in HDF5 (2.2.0) thread 8:
  #000: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5F.c line 490 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4109 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4045 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5Fint.c line 1084 in H5F__i

ok precip_2025: 2 regional files

========== DOWNLOAD SUMMARY ==========

Prices:
(6408, 2)
2000-07-17 00:00:00 to 2025-12-30 00:00:00

ONI:
1950-01-01 00:00:00 to 2026-09-01 00:00:00

MapSPAM:
(608865, 6)

CPC files:
north_america    tmax    : 36 files
north_america    tmin    : 36 files
north_america    precip  : 36 files
south_america    tmax    : 36 files
south_america    tmin    : 36 files
south_america    precip  : 36 files

Cell 5 completed.


HDF5-DIAG: Error detected in HDF5 (2.2.0) thread 5:
  #000: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5F.c line 490 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4109 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLcallback.c line 4045 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: /Users/runner/miniforge3/conda-bld/hdf5_1786232861201/work/src/H5Fint.c line 1084 in H5F__i

In [5]:
# ============================================================
# FINANCIAL DATA PROCESSING
# ============================================================

ROLL_SPECS = {
    "corn": {
        "roll": "cbot",
        "months": [3, 5, 7, 9, 12],
    },
    "soybeans": {
        "roll": "cbot",
        "months": [1, 3, 5, 7, 8, 9, 11],
    },
}

# Restrict the dataset to the two selected commodities
prices_daily = prices_daily[["corn", "soybeans"]].copy()
prices_daily.index = pd.to_datetime(prices_daily.index)
prices_daily = prices_daily.sort_index()

# ------------------------------------------------------------
# 1. Futures contract roll calendar
# ------------------------------------------------------------

def roll_day(trading_days, year, month, rule):
    """Identify the first trading day of the new futures contract."""

    in_month = trading_days[
        (trading_days.year == year) &
        (trading_days.month == month)
    ]

    if len(in_month) == 0:
        return None

    if rule == "cbot":
        after_15 = in_month[in_month.day >= 15]
        return after_15[0] if len(after_15) else None

    raise ValueError(f"Unknown roll rule: {rule}")


def roll_days(trading_days, rule, months):
    """Identify all contract roll days in the sample."""

    days = [
        roll_day(trading_days, year, month, rule)
        for year in trading_days.year.unique()
        for month in months
    ]

    return pd.DatetimeIndex([day for day in days if day is not None])


def expiry_days(trading_days, rolls):
    """Identify the last trading day before each contract roll."""

    positions = trading_days.get_indexer(rolls)
    valid_positions = positions[positions > 0]

    return trading_days[valid_positions - 1]


# ------------------------------------------------------------
# 2. Daily log returns
# ------------------------------------------------------------

daily_returns = {}

for commodity in ["corn", "soybeans"]:

    close = prices_daily[commodity].dropna()

    ret = np.log(close).diff()

    spec = ROLL_SPECS[commodity]

    rolls = roll_days(
        close.index,
        spec["roll"],
        spec["months"]
    )

    expiries = expiry_days(close.index, rolls)

    # Remove contract-switch and expiry-day returns
    ret.loc[ret.index.isin(rolls)] = np.nan
    ret.loc[ret.index.isin(expiries)] = np.nan

    daily_returns[commodity] = ret

daily_returns = pd.DataFrame(daily_returns)

# ------------------------------------------------------------
# 3. Weekly aggregation — Friday-ending weeks
# ------------------------------------------------------------

# Require at least three valid daily returns per week
weekly_returns = daily_returns.resample("W-FRI").sum(min_count=3)
weekly_returns = weekly_returns.loc[:END_DATE]

# ------------------------------------------------------------
# 4. Financial predictors and forecasting target
# ------------------------------------------------------------

financial_frames = []

for commodity in ["corn", "soybeans"]:

    r = weekly_returns[commodity]

    vol_4w = (
        daily_returns[commodity]
        .rolling(20, min_periods=15)
        .std()
        .resample("W-FRI")
        .last()
        * np.sqrt(252)
    )

    vol_12w = (
        daily_returns[commodity]
        .rolling(60, min_periods=45)
        .std()
        .resample("W-FRI")
        .last()
        * np.sqrt(252)
    )

    df = pd.DataFrame(index=weekly_returns.index)

    # Financial predictors
    df["ret_1w"] = r
    df["ret_4w"] = r.rolling(4).sum()
    df["ret_12w"] = r.rolling(12).sum()
    df["vol_4w"] = vol_4w
    df["vol_12w"] = vol_12w

    # Calendar seasonality
    month = df.index.month
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    # Forecasting target: following week's log return
    df["target_1w"] = r.shift(-1)

    df["commodity"] = commodity

    financial_frames.append(df)

financial_weekly = pd.concat(financial_frames)

financial_weekly = (
    financial_weekly
    .rename_axis("date")
    .reset_index()
    .set_index(["date", "commodity"])
    .sort_index()
)

print("Financial weekly dataset:", financial_weekly.shape)
print("\nObservations by commodity:")
print(financial_weekly.groupby("commodity").size())

print("\nDate range:")
print(financial_weekly.index.get_level_values("date").min())
print(financial_weekly.index.get_level_values("date").max())

print("\nSample:")
display(financial_weekly.head())


Financial weekly dataset: (2656, 8)

Observations by commodity:
commodity
corn        1328
soybeans    1328
dtype: int64

Date range:
2000-07-21 00:00:00
2025-12-26 00:00:00

Sample:


ret_1w  ret_4w  ret_12w  vol_4w  vol_12w  month_sin  \
date       commodity                                                          
2000-07-21 corn       0.046330     NaN      NaN     NaN      NaN  -0.500000   
           soybeans        NaN     NaN      NaN     NaN      NaN  -0.500000   
2000-07-28 corn      -0.040767     NaN      NaN     NaN      NaN  -0.500000   
           soybeans        NaN     NaN      NaN     NaN      NaN  -0.500000   
2000-08-04 corn      -0.008357     NaN      NaN     NaN      NaN  -0.866025   

                      month_cos  target_1w  
date       commodity                        
2000-07-21 corn       -0.866025  -0.040767  
           soybeans   -0.866025        NaN  
2000-07-28 corn       -0.866025  -0.008357  
           soybeans   -0.866025        NaN  
2000-08-04 corn       -0.500000  -0.022632

## B. Agricultural Production Weights

Weather conditions are not equally relevant across all geographical areas. We therefore aggregate weather observations using agricultural production weights.

Production data are obtained from MapSPAM 2010. For each commodity and production region, the geographical coordinates of MapSPAM production cells are mapped to the nearest CPC weather-grid cells. Production is then aggregated at the weather-grid-cell level and converted into normalised weights.

This approach gives greater importance to weather conditions in areas with higher agricultural production, rather than treating all geographical cells equally.

The resulting weights are used to construct production-weighted weather indicators in the next step.

In [ ]:
# ============================================================
# B - AGRICULTURAL PRODUCTION WEIGHTS
# Based on the original src/weights.py implementation
# ============================================================

# Commodity specifications required for geographical weighting
COMMODITY_SPECS = {
    "corn": {
        "crop": "maiz",
        "zones": {
            "US": {
                "iso3": ["USA"],
                "box": "north_america",
            },
        },
    },
    "soybeans": {
        "crop": "soyb",
        "zones": {
            "US": {
                "iso3": ["USA"],
                "box": "north_america",
            },
            "BRA": {
                "iso3": ["BRA"],
                "box": "south_america",
            },
            "ARG": {
                "iso3": ["ARG"],
                "box": "south_america",
            },
        },
    },
}


def to_180(lon):
    """Convert longitudes from [0, 360] to [-180, 180]."""
    return (np.asarray(lon) + 180) % 360 - 180


def cpc_grid(box):
    """Retrieve CPC grid-cell coordinates within a geographical box."""

    samples = sorted((DATA_RAW / "cpc" / box).glob("tmax_*.nc"))

    if not samples:
        raise FileNotFoundError(f"No CPC tmax files found for {box}")

    sample = samples[0]

    with xr.open_dataset(sample) as ds:
        return ds["lat"].values, to_180(ds["lon"].values)


def snap(values, grid):
    """Map coordinates to the nearest CPC grid-cell centre."""

    nearest = np.abs(values[:, None] - grid[None, :]).argmin(axis=1)
    return grid[nearest]


def crop_points(spam, crop, zone):
    """Extract MapSPAM production cells belonging to a geographical zone."""

    cells = spam[spam["iso3"].isin(zone["iso3"])]

    if "adm1" in zone:
        cells = cells[cells["adm1"].isin(zone["adm1"])]

    if "bbox" in zone:
        lat_min, lat_max, lon_min, lon_max = zone["bbox"]
        cells = cells[
            cells["lat"].between(lat_min, lat_max)
            & cells["lon"].between(lon_min, lon_max)
        ]

    return pd.DataFrame({
        "lat": cells["lat"],
        "lon": cells["lon"],
        "mass": cells[crop],
    })


def build_weights():
    """Build normalised production weights for each commodity and zone."""

    spam = pd.read_parquet(
        DATA_RAW / "spam" / "spam2010_production.parquet"
    )

    rows = []

    for name, spec in COMMODITY_SPECS.items():

        for zone_name, zone in spec["zones"].items():

            points = crop_points(spam, spec["crop"], zone)
            points = points[points["mass"] > 0]

            # Map production cells to the nearest CPC grid cells
            lats, lons = cpc_grid(zone["box"])

            points = points.assign(
                lat=snap(points["lat"].values, lats),
                lon=snap(points["lon"].values, lons),
            )

            # Aggregate production within each CPC grid cell
            cells = (
                points.groupby(["lat", "lon"], as_index=False)["mass"]
                .sum()
            )

            # Normalise production weights within each zone
            if cells["mass"].sum() <= 0:
                raise ValueError(f"No positive production mass for {name}, {zone_name}")
            cells["weight"] = cells["mass"] / cells["mass"].sum()

            cells["commodity"] = name
            cells["zone"] = zone_name
            cells["box"] = zone["box"]

            rows.append(cells)

            print(
                f"{name:10s} {zone_name:10s} "
                f"{len(cells):5d} cells"
            )

    columns = [
        "commodity", "zone", "box",
        "lat", "lon", "mass", "weight"
    ]

    return pd.concat(rows, ignore_index=True)[columns]


# Construct and save the production weights
weights = build_weights()

weights.to_parquet(
    DATA_PROCESSED / "weights.parquet"
)

print("\nProduction weights dataset:", weights.shape)

# Verify that weights sum to one within each commodity and zone
display(
    weights.groupby(["commodity", "zone"])["weight"].sum()
)

display(weights.head())

corn       US          2179 cells
soybeans   US          1594 cells
soybeans   BRA         1228 cells
soybeans   ARG          570 cells

Production weights dataset: (5571, 7)


commodity  zone
corn       US      1.0
soybeans   ARG     1.0
           BRA     1.0
           US      1.0
Name: weight, dtype: float64

,commodity,zone,box,lat,lon,mass,weight
0,corn,US,north_america,26.25,-98.75,691.4,0.000002
1,corn,US,north_america,26.25,-98.25,40099.6,0.000125
2,corn,US,north_america,26.25,-97.75,15541.7,0.000048
3,corn,US,north_america,26.75,-98.25,5304.0,0.000017
4,corn,US,north_america,27.25,-98.25,3726.4,0.000012



## C. Weather Data

Historical daily weather observations are obtained from the NOAA Climate Prediction Center (CPC). Three meteorological variables are used: daily maximum temperature (`tmax`), daily minimum temperature (`tmin`) and daily precipitation (`precip`).

The dataset covers 1990–2025 and includes two geographical regions: North America and South America. These regions cover the production areas selected for corn and soybeans in the United States, Brazil and Argentina.

Weather observations are linked to agricultural production through the geographical weights constructed in the previous section. This allows us to derive production-weighted weather indicators that reflect the meteorological conditions affecting the main production areas of each commodity.

The daily weather variables will subsequently be transformed into weekly indicators and used to construct the Weather Stress Index (WSI).


In [9]:
# ============================================================
# C. Weather Data: Daily indicators and weekly aggregation
# ============================================================

HEAVY_RAIN_MM = 20.0

WEEKLY_AGG = {
    "tmax": "mean",
    "tmin": "mean",
    "tavg": "mean",
    "precip": "sum",
    "et0": "sum",
    "water_balance": "sum",
    "heavy_rain_days": "sum",
    "edd": "sum",
    "frost_dd": "sum",
}

WEATHER_SPECS = {
    "corn": {
        "crop": "maiz",
        "heat_t": 30.0,
        "frost_t": 0.0,
    },
    "soybeans": {
        "crop": "soyb",
        "heat_t": 30.0,
        "frost_t": 0.0,
    },
}


def read_cells(box, variable, lats, lons):
    """Read daily CPC observations for the selected grid cells."""
    years = range(
        pd.Timestamp(WEATHER_START).year,
        pd.Timestamp(END_DATE).year + 1
    )

    chunks = []

    for year in years:
        path = DATA_RAW / "cpc" / box / f"{variable}_{year}.nc"

        with xr.open_dataset(path) as ds:
            ds = ds.assign_coords(lon=to_180(ds["lon"].values))

            values = ds[variable].sel(
                lat=xr.DataArray(lats, dims="cell"),
                lon=xr.DataArray(lons, dims="cell")
            )

            chunks.append(values.to_pandas())

    return pd.concat(chunks)


def extraterrestrial_radiation(lat_deg, day_of_year):
    """FAO-56 extraterrestrial radiation, expressed in mm/day."""
    phi = np.radians(lat_deg)[None, :]
    j = day_of_year[:, None]

    dr = 1 + 0.033 * np.cos(2 * np.pi * j / 365)
    delta = 0.409 * np.sin(2 * np.pi * j / 365 - 1.39)

    ws = np.arccos(
        np.clip(-np.tan(phi) * np.tan(delta), -1, 1)
    )

    ra = (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta)
        + np.cos(phi) * np.cos(delta) * np.sin(ws)
    )

    return 0.408 * ra


def daily_variables(tmax, tmin, precip, lats, spec):
    """Calculate daily weather indicators at each grid cell."""

    tavg = (tmax + tmin) / 2

    ra = extraterrestrial_radiation(
        lats,
        tmax.index.dayofyear.values
    )

    # Hargreaves reference evapotranspiration (FAO-56).
    et0 = (
        0.0023
        * ra
        * (tavg + 17.8)
        * np.sqrt((tmax - tmin).clip(lower=0))
    )

    out = {
        "tmax": tmax,
        "tmin": tmin,
        "tavg": tavg,
        "precip": precip,
        "et0": et0,
        "water_balance": precip - et0,
        "heavy_rain_days": (
            (precip >= HEAVY_RAIN_MM)
            .astype(float)
            .where(precip.notna())
        ),
        "edd": (tmax - spec["heat_t"]).clip(lower=0),
        "frost_dd": (spec["frost_t"] - tmin).clip(lower=0),
    }

    return out


def weighted_mean(values, weights):
    """Production-weighted average across grid cells."""
    w = pd.DataFrame(
        np.broadcast_to(weights, values.shape),
        index=values.index,
        columns=values.columns
    ).where(values.notna())

    return (values * w).sum(axis=1) / w.sum(axis=1)


def zone_weekly(commodity, zone, cells):
    """Construct weekly weather indicators for one production zone."""

    spec = WEATHER_SPECS[commodity]

    box = cells["box"].iloc[0]

    lats = cells["lat"].values
    lons = cells["lon"].values
    weights_array = cells["weight"].values

    raw = {
        variable: read_cells(box, variable, lats, lons)
        for variable in ["tmax", "tmin", "precip"]
    }

    daily = daily_variables(
        raw["tmax"],
        raw["tmin"],
        raw["precip"],
        lats,
        spec
    )

    # Apply production weights after calculating daily indicators.
    zone_daily = pd.DataFrame({
        name: weighted_mean(df, weights_array)
        for name, df in daily.items()
    })

    zone_daily = zone_daily.loc[WEATHER_START:END_DATE]

    # Aggregate daily observations into Friday-ending weeks.
    weekly = zone_daily.resample(WEEK_RULE).agg(WEEKLY_AGG)
    return weekly.loc[:END_DATE]


def build_weather():
    """Build the weekly weather dataset for all commodity zones."""

    weights = pd.read_parquet(
        DATA_PROCESSED / "weights.parquet"
    )

    frames = []

    for (commodity, zone), cells in weights.groupby(
        ["commodity", "zone"],
        sort=False
    ):

        weekly = zone_weekly(commodity, zone, cells)

        long = (
            weekly.rename_axis("date")
            .reset_index()
            .melt(
                id_vars="date",
                var_name="variable",
                value_name="value"
            )
        )

        frames.append(
            long.assign(
                commodity=commodity,
                zone=zone
            )
        )

        print(
            f"{commodity:10s} {zone:10s} "
            f"{len(weekly)} weeks",
            flush=True
        )

    return pd.concat(
        frames,
        ignore_index=True
    )[["date", "commodity", "zone", "variable", "value"]]


weather_weekly = build_weather()

weather_weekly.to_parquet(
    DATA_PROCESSED / "weather_weekly.parquet"
)

print("\nWeekly weather dataset:", weather_weekly.shape)
display(weather_weekly.head())

/var/folders/7j/4w_vw4pd0wj6kyp7y235vt3c0000gn/T/ipykernel_13831/1192556871.py:85: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  tmax.index.dayofyear.values


corn       US         1878 weeks


/var/folders/7j/4w_vw4pd0wj6kyp7y235vt3c0000gn/T/ipykernel_13831/1192556871.py:85: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  tmax.index.dayofyear.values


soybeans   US         1878 weeks


/var/folders/7j/4w_vw4pd0wj6kyp7y235vt3c0000gn/T/ipykernel_13831/1192556871.py:85: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  tmax.index.dayofyear.values


soybeans   BRA        1878 weeks


/var/folders/7j/4w_vw4pd0wj6kyp7y235vt3c0000gn/T/ipykernel_13831/1192556871.py:85: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  tmax.index.dayofyear.values


soybeans   ARG        1878 weeks

Weekly weather dataset: (67608, 5)


,date,commodity,zone,variable,value
0,1990-01-05,corn,US,tmax,4.737534
1,1990-01-12,corn,US,tmax,6.944887
2,1990-01-19,corn,US,tmax,7.399158
3,1990-01-26,corn,US,tmax,5.751783
4,1990-02-02,corn,US,tmax,5.073322


## III. Weather Stress Index Construction

### A. Weather Stress Components

We construct five weather stress components for each commodity and production region. All components are oriented so that higher values indicate greater weather stress.

**1. Physical stress indicators**

The five indicators are defined as follows:

- **Heat:** extreme degree days above 30°C:

$$
Heat_{r,t} = \sum_{d \in t} \max(T^{max}_{r,d} - 30, 0)
$$

- **Warm nights:** weekly average minimum temperature:

$$
WarmNights_{r,t} =
\frac{1}{N_t}\sum_{d \in t} T^{min}_{r,d}
$$

- **Heavy rain:** number of days with precipitation of at least 20 mm:

$$
HeavyRain_{r,t} =
\sum_{d \in t} \mathbb{1}(P_{r,d} \geq 20)
$$

- **Frost:** cumulative frost degree days below 0°C:

$$
Frost_{r,t} =
\sum_{d \in t} \max(-T^{min}_{r,d}, 0)
$$

- **Dryness:** negative four-week cumulative water balance, where water balance is precipitation minus reference evapotranspiration:

$$
Dryness_{r,t} =
-\sum_{j=0}^{3}(P_{r,t-j} - ET0_{r,t-j})
$$

Here, $r$ denotes a production region, $t$ a week and $d$ a day within that week.

**2. Historical seasonal standardisation**

The four continuous indicators (heat, warm nights, heavy rain and dryness) are converted into seasonal anomalies using a past-only z-score:

$$
Z_{r,t} =
\frac{X_{r,t} - \mu_{r,t}^{past}}
{\sigma_{r,t}^{past}}
$$

The historical mean and standard deviation are calculated using observations from the same week of the year, within a ±2-week window, and from previous years only. At least five distinct historical years are required.

This expanding-window procedure prevents future weather observations from entering the historical reference distribution. If the historical standard deviation is zero, the anomaly is set to zero. Frost is retained in its original degree-day units rather than standardised.

**3. Growing-season adjustment and regional aggregation**

Weather stress is considered relevant only during the crop's growing season. Components are therefore set to zero outside the growing season, while missing observations during the growing season remain missing.

Finally, regional components are aggregated using their respective shares of total production:

$$
C_{c,t} = \sum_{r \in R_c} s_{c,r} C_{r,t}
$$

where the production share of region $r$ for commodity $c$ is:

$$
s_{c,r} =
\frac{Production_{c,r}}
{\sum_{k \in R_c} Production_{c,k}}
$$

This produces five weekly, production-weighted weather stress components for each commodity.

In [11]:
# ============================================================
# A. Construction of weekly weather stress components
# ============================================================

CLIMATOLOGY_HALF_WINDOW = 2
MIN_PAST_YEARS = 5

# Growing seasons by commodity and production region.
CROP_SEASONS = {
    "corn": {
        "US": [5, 6, 7, 8, 9],
    },
    "soybeans": {
        "US": [6, 7, 8, 9],
        "BRA": [11, 12, 1, 2],
        "ARG": [12, 1, 2, 3],
    },
}


def expanding_anomaly(x, in_season):
    """
    Calculate a past-only seasonal z-score.

    Historical observations are selected from the same week of the year,
    within a +/- 2-week window, and from previous years only.
    """

    years = x.index.year.to_numpy()
    weeks = np.minimum(
        x.index.isocalendar().week.to_numpy(),
        52
    )
    active = in_season.to_numpy(dtype=bool)

    z = pd.Series(np.nan, index=x.index, dtype=float)

    for year in np.unique(years):

        # Use only valid in-season observations from previous years.
        past_mask = (
            (years < year)
            & active
            & x.notna().to_numpy()
        )

        if len(np.unique(years[past_mask])) < MIN_PAST_YEARS:
            continue

        mean = np.full(53, np.nan)
        std = np.full(53, np.nan)

        past_values = x.to_numpy()[past_mask]
        past_weeks = weeks[past_mask]
        past_years = years[past_mask]

        for week in range(1, 53):

            gap = np.abs(past_weeks - week)

            # Circular distance handles the beginning/end of the year.
            in_window = (
                np.minimum(gap, 52 - gap)
                <= CLIMATOLOGY_HALF_WINDOW
            )

            values = past_values[in_window]
            observation_years = past_years[in_window]

            if len(np.unique(observation_years)) < MIN_PAST_YEARS:
                continue

            mean[week] = np.mean(values)
            std[week] = np.std(values, ddof=1)

        current_mask = (years == year) & active
        current_weeks = weeks[current_mask]
        current_values = x.to_numpy()[current_mask]

        m = mean[current_weeks]
        s = std[current_weeks]

        valid_mean = ~np.isnan(m)
        result = np.full(len(current_values), np.nan)

        positive_std = valid_mean & (s > 0)
        zero_std = valid_mean & (s == 0)

        result[positive_std] = (
            current_values[positive_std] - m[positive_std]
        ) / s[positive_std]

        result[zero_std] = 0.0

        z.iloc[np.where(current_mask)[0]] = result

    # Off-season values are explicitly set to zero.
    z.loc[~in_season] = 0.0

    return z


def zone_components(weather, in_season):
    """Construct stress components for one commodity-region."""

    # Physical weather indicators.
    raw = pd.DataFrame({
        "heat": weather["edd"],
        "warm_nights": weather["tmin"],
        "heavy_rain": weather["heavy_rain_days"],
        "frost": weather["frost_dd"],
    })

    # Four-week cumulative water balance, oriented so higher = drier.
    raw["dryness"] = -(
        weather["water_balance"]
        .rolling(window=4, min_periods=4)
        .sum()
        .where(in_season)
    )

    # Standardise continuous components using historical observations.
    components = pd.DataFrame({
        column: expanding_anomaly(raw[column], in_season)
        for column in raw.columns
    })

    # Keep frost in its original degree-day units.
    components["frost"] = raw["frost"]
    components.loc[~in_season, "frost"] = 0.0

    return components


def build_components():
    """Build production-weighted weekly components for each commodity."""

    weather = pd.read_parquet(
        DATA_PROCESSED / "weather_weekly.parquet"
    )

    weights = pd.read_parquet(
        DATA_PROCESSED / "weights.parquet"
    )

    # Total production by commodity and geographical region.
    zone_mass = (
        weights.groupby(["commodity", "zone"])["mass"].sum()
    )

    available = set(
        weather.groupby(["commodity", "zone"]).groups
    )

    frames = []

    for commodity, zones in CROP_SEASONS.items():

        # Ensure that no production region is silently omitted.
        missing_zones = [
            zone
            for zone in zones
            if (commodity, zone) not in available
        ]

        if missing_zones:
            raise ValueError(
                f"{commodity}: missing weather data for {missing_zones}"
            )

        # Production shares across the commodity's regions.
        shares = zone_mass.loc[commodity]
        shares = shares / shares.sum()

        total = None

        for zone, months in zones.items():

            # Convert long-format weather data into a weekly wide table.
            w = (
                weather.query(
                    "commodity == @commodity and zone == @zone"
                )
                .pivot(
                    index="date",
                    columns="variable",
                    values="value"
                )
                .sort_index()
            )

            # Define the regional growing season.
            in_season = pd.Series(
                w.index.month.isin(months),
                index=w.index,
                dtype=bool
            )

            comp = zone_components(w, in_season)

            # Weight regional components by production share.
            weighted_comp = comp * shares[zone]

            # Preserve missing values rather than treating them as zero.
            total = (
                weighted_comp
                if total is None
                else total + weighted_comp
            )

        frames.append(
            total.assign(commodity=commodity)
        )

        print(
            f"{commodity:10s} {list(total.columns)}",
            flush=True
        )

    out = (
        pd.concat(frames)
        .rename_axis("date")
        .reset_index()
    )

    return (
        out.set_index(["date", "commodity"])
        .sort_index()
    )


components_weekly = build_components()

components_weekly.to_parquet(
    DATA_PROCESSED / "components_weekly.parquet"
)

print("\nComponents dataset:", components_weekly.shape)
print("\nDate range:")
print(
    components_weekly.index.get_level_values("date").min(),
    "to",
    components_weekly.index.get_level_values("date").max()
)

display(components_weekly.head())

corn       ['heat', 'warm_nights', 'heavy_rain', 'frost', 'dryness']
soybeans   ['heat', 'warm_nights', 'heavy_rain', 'frost', 'dryness']

Components dataset: (3756, 5)

Date range:
1990-01-05 00:00:00 to 2025-12-26 00:00:00


heat  warm_nights  heavy_rain  frost  dryness
date       commodity                                               
1990-01-05 corn        0.0          0.0         0.0    0.0      0.0
           soybeans    NaN          NaN         NaN    0.0      NaN
1990-01-12 corn        0.0          0.0         0.0    0.0      0.0
           soybeans    NaN          NaN         NaN    0.0      NaN
1990-01-19 corn        0.0          0.0         0.0    0.0      0.0


### B. Missing Values and Historical Data Availability

The resulting dataset contains 110 missing observations for corn and 202 for soybeans in the four standardised components (heat, warm nights, heavy rain and dryness), due to the minimum five-year historical requirement of the expanding climatology.

These missing values are therefore a consequence of the chosen methodology rather than missing weather observations. They are preserved at this stage and will be handled when constructing the forecasting dataset.



### C. Composite Weather Stress Index

The five weather components constructed in the previous section are combined into a single Weather Stress Index (WSI), providing an overall measure of agricultural weather stress.

The index construction is inspired by the Climate Physical Risk Index (CPRI) methodology proposed by Guo, Ji and Zhang (2024). However, the approach adopted in this project is an adaptation to agricultural commodities and does not constitute an exact replication of the original CPRI methodology.

**Component normalisation**

Each weather component is independently rescaled between 0 and 100 using an expanding min–max normalisation. For component $j$ at week $t$:

$$
S_{j,t} =
100 \times
\frac{X_{j,t}-X_{j,\min}^{(t-1)}}
{X_{j,\max}^{(t-1)}-X_{j,\min}^{(t-1)}}
$$

where:
- $X_{j,t}$ is the observed value of component $j$ at week $t$.
- $X_{j,\min}^{(t-1)}$ and $X_{j,\max}^{(t-1)}$ are respectively the historical minimum and maximum calculated using observations strictly prior to week $t$.

Only complete, in-season observations are included when estimating these historical ranges. This past-only procedure prevents future information from entering the normalisation.

**Composite index aggregation**

The final WSI is calculated as the equally weighted arithmetic average of the five normalised components:

$$
WSI_t = \frac{1}{5}\sum_{j=1}^{5} S_{j,t}
$$

The index ranges from 0 to 100. It is assigned a value of zero outside the growing season. In-season observations with at least one missing component remain missing rather than being imputed.


In [14]:
# ============================================================
# C. Composite Weather Stress Index
# ============================================================
def get_season_mask(commodity, dates):
    """Return the union of configured production-zone growing seasons."""

    season_months = {
        "corn": {
            "US": [5, 6, 7, 8, 9],
        },
        "soybeans": {
            "US": [6, 7, 8, 9],
            "BRA": [11, 12, 1, 2],
            "ARG": [12, 1, 2, 3],
        },
    }

    active_months = set()

    for months in season_months[commodity].values():
        active_months.update(months)

    return pd.Series(
        dates.month.isin(active_months),
        index=dates,
        name="in_season"
    )


def cpri_style_index(components, in_season):
    """Build a past-only min-max index with equal component weights."""

    # Only complete in-season observations enter the historical ranges.
    complete = components.notna().all(axis=1)

    historical = components.where(in_season & complete)

    # Expanding historical minimum and maximum.
    # shift(1) prevents the current observation from defining its own range.
    historical_min = (
        historical.expanding(min_periods=1).min().shift(1)
    )
    historical_max = (
        historical.expanding(min_periods=1).max().shift(1)
    )

    historical_range = historical_max - historical_min

    # Min-max normalisation.
    denominator = historical_range.where(historical_range > 0)

    scaled = (
        100.0
        * (components - historical_min)
        / denominator
    )

    # Handle a zero historical range.
    zero_range = (
        historical_range.eq(0)
        & historical_min.notna()
    )

    scaled = scaled.mask(
        zero_range & components.le(historical_min),
        0.0
    )
    scaled = scaled.mask(
        zero_range & components.gt(historical_min),
        100.0
    )

    # Keep the scale bounded between 0 and 100.
    scaled = scaled.clip(lower=0.0, upper=100.0)

    # Require all components to be available.
    valid = (
        in_season
        & complete
        & scaled.notna().all(axis=1)
    )

    # Equal-weighted arithmetic average.
    index = scaled.mean(axis=1).where(valid)

    # Off-season values are not applicable.
    index.loc[~in_season] = 0.0

    return index.rename("wsi_cpri")


# Process each commodity independently.
wsi_frames = []

for commodity, raw in components_weekly.groupby(
    level="commodity",
    sort=True
):
    if commodity not in COMMODITY_SPECS:
        continue

    x = (
        raw.droplevel("commodity")
        .sort_index()
        .copy()
    )

    x.index = pd.DatetimeIndex(x.index, name="date")

    # Remove components that are entirely missing.
    x = x.dropna(axis=1, how="all")

    if x.empty:
        print(f"{commodity}: skipped (no available components)")
        continue

    in_season = get_season_mask(commodity, x.index)

    wsi = cpri_style_index(x, in_season)

    frame = pd.DataFrame(
        {
            "wsi_cpri": wsi,
            "in_season": in_season.astype("int8")
        },
        index=x.index
    )

    frame["commodity"] = commodity
    wsi_frames.append(frame)

    n_valid = int(wsi.loc[in_season].notna().sum())

    print(
        f"{commodity}: "
        f"{n_valid} valid in-season index observations."
    )


wsi_weekly = (
    pd.concat(wsi_frames)
    .rename_axis("date")
    .reset_index()
    .set_index(["date", "commodity"])
    .sort_index()
)

wsi_weekly.to_parquet(DATA_PROCESSED / "wsi_cpri_weekly.parquet")

print("\nWSI dataset:", wsi_weekly.shape)
display(wsi_weekly.head())

corn: 676 valid in-season index observations.
soybeans: 1202 valid in-season index observations.

WSI dataset: (3756, 2)


wsi_cpri  in_season
date       commodity                     
1990-01-05 corn            0.0          0
           soybeans        NaN          1
1990-01-12 corn            0.0          0
           soybeans        NaN          1
1990-01-19 corn            0.0          0

### D. Alternative PCA-Based Weather Stress Index

As an alternative to the equal-weighted WSI, we investigate a data-driven aggregation method based on Principal Component Analysis (PCA).

The objective is to identify the main direction of variation among the five weather components and use the first principal component (PC1) as an alternative measure of weather stress.

Following the approach explored in Ma, Zhou and Li (2024), PCA is estimated separately for each commodity and re-estimated every January using an expanding historical window. Only complete, in-season observations from previous years are used for estimation, with a minimum requirement of 104 observations.

The estimated component loadings are then used to project the observations of the current year. This procedure ensures that future observations are not used to estimate the PCA weights.

The resulting index and yearly loadings are retained for comparison and exploratory analysis.

In [17]:
# ============================================================
# D. Alternative PCA-based Weather Stress Index
# ============================================================
from sklearn.decomposition import PCA
MIN_FIT_WEEKS = 104  # Minimum number of previous in-season weeks


def pca_index(components):
    """
    First principal component, refitted each January
    using all previous complete in-season observations.
    """

    index = pd.Series(np.nan, index=components.index)
    loadings = []

    # Retain only complete observations.
    clean = components.dropna()

    # Exclude off-season observations from PCA estimation.
    active = clean[(clean != 0).any(axis=1)]

    for year in sorted(clean.index.year.unique()):

        # Expanding training window using previous years only.
        train = active[active.index.year < year]

        if len(train) < MIN_FIT_WEEKS:
            continue

        pca = PCA(n_components=1).fit(train.values)

        w = pca.components_[0]

        # Ensure a positive sum of loadings.
        w = w if w.sum() >= 0 else -w

        # Project current-year observations onto the first PC.
        this_year = clean[clean.index.year == year]

        index[this_year.index] = (
            (this_year.values - pca.mean_) @ w
        )

        # Off-season observations receive a zero index.
        off_season = (this_year == 0).all(axis=1)
        index[this_year.index[off_season]] = 0.0

        # Store yearly loadings and explained variance.
        loadings.append({
            "year": year,
            "explained_var": pca.explained_variance_ratio_[0],
            **dict(zip(components.columns, w))
        })

    return index.rename("wsi_pca"), pd.DataFrame(loadings)


# Process each commodity independently.
pca_frames = []
pca_loading_frames = []

for commodity, raw in components_weekly.groupby(
    level="commodity",
    sort=True
):

    x = (
        raw.droplevel("commodity")
        .sort_index()
        .copy()
    )

    x.index = pd.DatetimeIndex(x.index, name="date")
    x = x.dropna(axis=1, how="all")

    if x.empty:
        continue

    pca, loadings = pca_index(x)

    frame = pd.DataFrame(
        {"wsi_pca": pca},
        index=x.index
    )

    frame["commodity"] = commodity
    pca_frames.append(frame)

    loadings["commodity"] = commodity
    pca_loading_frames.append(loadings)

    print(
        f"{commodity}: "
        f"{pca.notna().sum()} valid PCA index observations."
    )


wsi_pca_weekly = (
    pd.concat(pca_frames)
    .rename_axis("date")
    .reset_index()
    .set_index(["date", "commodity"])
    .sort_index()
)

pca_loadings = pd.concat(
    pca_loading_frames,
    ignore_index=True
)

# Save PCA index and yearly loadings.
wsi_pca_weekly.to_parquet(
    DATA_PROCESSED / "wsi_pca_weekly.parquet"
)

pca_loadings.to_parquet(
    DATA_PROCESSED / "pca_loadings.parquet"
)

print("\nPCA index dataset:", wsi_pca_weekly.shape)
print("PCA loadings dataset:", pca_loadings.shape)

display(wsi_pca_weekly.head())
display(pca_loadings.head())

corn: 1356 valid PCA index observations.
soybeans: 1458 valid PCA index observations.

PCA index dataset: (3756, 1)
PCA loadings dataset: (54, 8)


wsi_pca
date       commodity         
1990-01-05 corn           NaN
           soybeans       NaN
1990-01-12 corn           NaN
           soybeans       NaN
1990-01-19 corn           NaN

,year,explained_var,heat,warm_nights,heavy_rain,frost,dryness,commodity
0,2000,0.596680,0.733045,0.622423,-0.152125,-0.037798,0.225084,corn
1,2001,0.592074,0.730745,0.617532,-0.164296,-0.038099,0.237109,corn
2,2002,0.596918,0.726032,0.623532,-0.160054,-0.034275,0.239359,corn
3,2003,0.591525,0.711217,0.621731,-0.181049,-0.042788,0.270207,corn
4,2004,0.563913,0.702898,0.606050,-0.201570,-0.037902,0.310756,corn


## E. Assessment of the PCA-Based Index

PCA was investigated as an alternative method for aggregating the five weather components. The first principal component explains, on average, 51.0% of the total variance for corn and 49.5% for soybeans.

However, the estimated loadings reveal an important limitation. For both commodities, heat, warm nights and dryness have positive average loadings, whereas heavy rainfall and frost have negative average loadings.

This reflects the statistical objective of PCA: identifying the direction that maximises explained variance, rather than constructing an index in which all adverse weather conditions contribute positively to overall stress. As a result, the first principal component can combine weather indicators in ways that are difficult to reconcile with their individual agricultural interpretations.

Additional PCA specifications were also explored in the original analysis, including standardisation of the components and alternative estimation windows. These experiments were used to assess the robustness and interpretability of the PCA-based approach.

Ultimately, we do not retain the PCA index in the forecasting models. Instead, we use the CPRI-inspired equal-weighted WSI, which provides a more transparent aggregation rule: each weather component is independently normalised using historical information and contributes equally to the final index.

## IV. Modelling Dataset

### A. Construction of the Modelling Dataset

The financial predictors, the composite Weather Stress Index and the individual weather components are merged into a single weekly panel for corn and soybeans.

To avoid look-ahead bias, all weather variables are lagged by one weekly observation within each commodity. This ensures that the forecasting models only use weather information available at the previous Friday when predicting the next week's futures return.

Three feature specifications are constructed to assess the incremental predictive value of weather information:

- **M1 – Financial:** past returns, rolling volatility and cyclical month indicators.
- **M2 – WSI:** financial predictors augmented with the composite Weather Stress Index and the growing-season indicator.
- **M3 – Components:** financial predictors augmented with the five individual weather components and the growing-season indicator.

The PCA-based index is excluded following the assessment presented in the previous section. The resulting dataset contains the target variable and all predictors required for the subsequent out-of-sample forecasting analysis.

In [20]:
# ============================================================
# A. Construction of the Modelling Dataset
# ============================================================

WEATHER_COMPONENTS = [
    "heat",
    "warm_nights",
    "heavy_rain",
    "frost",
    "dryness",
]

# Combine the individual weather components and the composite WSI.
weather = components_weekly[WEATHER_COMPONENTS].join(
    wsi_weekly[["wsi_cpri", "in_season"]],
    how="outer",
)

# Lag all weather information by one week, separately for each commodity.
weather = (
    weather.groupby(level="commodity", sort=False)
    .shift(WEATHER_LAG)
)

# Merge financial predictors with lagged weather information.
model_dataset = financial_weekly.join(
    weather,
    how="left",
)

# Retain the selected commodities and the 2000–2025 sample.
model_dataset = model_dataset.loc[
    model_dataset.index.get_level_values("commodity").isin(COMMODITIES)
].copy()

model_dataset = model_dataset.loc[
    model_dataset.index.get_level_values("date").year <= 2025
]

# Keep only variables required for forecasting.
selected_columns = (
    ["target_1w"]
    + MARKET_FEATURES
    + ["wsi_cpri"]
    + WEATHER_COMPONENTS
    + ["in_season"]
)

model_dataset = model_dataset[selected_columns].sort_index()

# Preserve missing values while using a nullable integer for the season indicator.
model_dataset["in_season"] = model_dataset["in_season"].astype("Int64")

# Save the modelling dataset.
model_dataset.to_parquet(DATA_PROCESSED / "model_dataset.parquet")

print("Modelling dataset:", model_dataset.shape)
print(
    "Date range:",
    model_dataset.index.get_level_values("date").min(),
    "to",
    model_dataset.index.get_level_values("date").max(),
)
print("\nObservations by commodity:")
display(model_dataset.groupby(level="commodity").size())

print("\nMissing values:")
display(model_dataset.isna().sum().to_frame("missing observations"))

display(model_dataset.head())

Modelling dataset: (2656, 15)
Date range: 2000-07-21 00:00:00 to 2025-12-26 00:00:00

Observations by commodity:


commodity
corn        1328
soybeans    1328
dtype: int64


Missing values:


,missing observations
target_1w,18
ret_1w,17
ret_4w,36
ret_12w,77
vol_4w,22
vol_12w,40
month_sin,0
month_cos,0
wsi_cpri,2
heat,2


target_1w    ret_1w  ret_4w  ret_12w  vol_4w  vol_12w  \
date       commodity                                                          
2000-07-21 corn       -0.040767  0.046330     NaN      NaN     NaN      NaN   
           soybeans         NaN       NaN     NaN      NaN     NaN      NaN   
2000-07-28 corn       -0.008357 -0.040767     NaN      NaN     NaN      NaN   
           soybeans         NaN       NaN     NaN      NaN     NaN      NaN   
2000-08-04 corn       -0.022632 -0.008357     NaN      NaN     NaN      NaN   

                      month_sin  month_cos   wsi_cpri      heat  warm_nights  \
date       commodity                                                           
2000-07-21 corn       -0.500000  -0.866025  31.792846  0.246828     1.005744   
           soybeans   -0.500000  -0.866025  34.806873  0.091584     0.381973   
2000-07-28 corn       -0.500000  -0.866025  18.799441 -0.636284    -1.066226   
           soybeans   -0.500000  -0.866025  25.969741 -0.279360    -0.464455   
2000-08-04 corn       -0.866025  -0.500000  21.273625 -0.622489    -0.647723   

                      heavy_rain         frost   dryness  in_season  
date       commodity                                                 
2000-07-21 corn         0.285002  7.959012e-08 -0.560231          1  
           soybeans    -0.014963  0.000000e+00 -0.286089          1  
2000-07-28 corn        -0.754997  9.722368e-04 -0.359479          1  
           soybeans    -0.267722  1.364969e-03 -0.121356          1  
2000-08-04 corn        -1.345929  0.000000e+00  0.353208          1

### B. Expanding-Window Out-of-Sample Forecasting

Forecasts are generated using an expanding-window approach. The first out-of-sample year is 2010. For each subsequent year, models are estimated using all available observations prior to that year's first observation and then used to predict returns throughout the year.

The estimation window therefore expands over time, allowing the models to incorporate all previously available information while preserving the chronological order of the observations.

Three machine learning models are considered: Ridge Regression, Random Forest and HistGradientBoosting. Their hyperparameters follow the specifications established in the original project. A historical-mean forecast serves as the benchmark.

Each model is estimated separately for the three feature specifications (M1, M2 and M3). Preprocessing steps, including missing-value imputation and standardisation where applicable, are fitted within each training window to prevent look-ahead bias.

The resulting forecasts are stored alongside realised returns for the subsequent out-of-sample performance evaluation.

In [24]:

# ============================================================
# B. Expanding-Window Out-of-Sample Forecasting
# ============================================================

# ------------------------------------------------------------
# 1. Define the models 
# ------------------------------------------------------------

def get_models():

    return {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10.0)),
        ]),

        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=300,
                max_depth=5,
                min_samples_leaf=10,
                max_features=0.8,
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),

        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=SEED,
            )),
        ]),
    }


# ------------------------------------------------------------
# 2. Define the expanding-window forecasting procedure
# ------------------------------------------------------------

def run_walk_forward(df, commodity):

    # Construct a common sample across all feature specifications.
    all_features = list(dict.fromkeys(
        MARKET_FEATURES + WSI_FEATURES + COMPONENT_FEATURES
    ))

    data = (
        df.xs(commodity, level="commodity")
        .copy()
        .sort_index()
        .dropna(subset=["target_1w"] + all_features)
    )

    forecast_frames = []

    # Identify all out-of-sample years.
    test_years = sorted(
        data.index[data.index.year >= FIRST_TEST_YEAR].year.unique()
    )

    for year in test_years:

        # Split the data chronologically.
        test_data = data.loc[data.index.year == year]

        first_test_date = test_data.index.min()
        train_data = data.loc[data.index < first_test_date]

        y_train = train_data["target_1w"]

        # Historical-mean benchmark.
        historical_mean = y_train.mean()

        benchmark = pd.DataFrame({
            "date": test_data.index,
            "commodity": commodity,
            "year": year,
            "model": "Historical Mean",
            "feature_set": "Benchmark",
            "actual": test_data["target_1w"].values,
            "forecast": historical_mean,
        })

        forecast_frames.append(benchmark)

        # Instantiate models once per year.
        models = get_models()

        # Estimate models for each feature specification.
        for feature_name, features in FEATURE_SETS.items():

            X_train = train_data[features]
            X_test = test_data[features]

            for model_name, model in models.items():

                model.fit(X_train, y_train)

                predictions = model.predict(X_test)

                result = pd.DataFrame({
                    "date": test_data.index,
                    "commodity": commodity,
                    "year": year,
                    "model": model_name,
                    "feature_set": feature_name,
                    "actual": test_data["target_1w"].values,
                    "forecast": predictions,
                })

                forecast_frames.append(result)

    return pd.concat(forecast_frames, ignore_index=True)


# ------------------------------------------------------------
# 3. Run the forecasting procedure for both commodities
# ------------------------------------------------------------

forecasts = pd.concat(
    [
        run_walk_forward(model_dataset, commodity)
        for commodity in COMMODITIES
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Format and save the forecast results
# ------------------------------------------------------------

forecasts = (
    forecasts
    .set_index(["date", "commodity", "model", "feature_set"])
    .sort_index()
)

forecasts.to_parquet(RESULTS / "oos_forecasts.parquet")


# ------------------------------------------------------------
# 5. Display forecasting results and diagnostics
# ------------------------------------------------------------

print("Forecast dataset:", forecasts.shape)

print("\nForecasting period:")
print(
    forecasts.index.get_level_values("date").min(),
    "to",
    forecasts.index.get_level_values("date").max()
)

print("\nNumber of forecasts by commodity:")
display(forecasts.groupby(level="commodity").size())

print("\nNumber of forecasts by model and feature set:")
display(
    forecasts.reset_index()
    .groupby(["model", "feature_set"])
    .size()
    .unstack(fill_value=0)
)


Forecast dataset: (16550, 3)

Forecasting period:
2010-01-01 00:00:00 to 2025-12-19 00:00:00

Number of forecasts by commodity:


commodity
corn        8210
soybeans    8340
dtype: int64


Number of forecasts by model and feature set:


feature_set,Benchmark,M1_Financial,M2_WSI,M3_Components
model,,,,
HistGradientBoosting,0,1655,1655,1655
Historical Mean,1655,0,0,0
Random Forest,0,1655,1655,1655
Ridge,0,1655,1655,1655


**Forecasting Procedure – Summary**

The expanding-window procedure produces out-of-sample forecasts for both commodities from 2010 to 2025. Models are re-estimated annually using all observations available before the beginning of each test year.

For each commodity, all feature specifications are evaluated on the same common sample. The historical-mean benchmark and the three machine learning models are evaluated using identical test observations.

The resulting forecast dataset contains the realised returns, predicted returns, model identifiers and feature specifications. It is saved for the subsequent performance evaluation.

### C. Out-of-Sample Forecast Evaluation

The predictive performance of the models is assessed using four complementary metrics: out-of-sample $R^2$, Root Mean Squared Error (RMSE), Mean Absolute Error (MAE) and directional accuracy.

The out-of-sample $R^2$ measures the improvement in squared forecast errors relative to the expanding historical-mean benchmark. Positive values indicate better predictive performance than the benchmark, while negative values indicate underperformance.

RMSE and MAE measure the magnitude of forecasting errors, with lower values indicating better accuracy. Directional accuracy measures the proportion of forecasts that correctly predict the sign of the realised weekly return.

Metrics are calculated over the full out-of-sample period and separately for each calendar year. This provides both an overall assessment of predictive performance and an indication of its stability over time.

All metrics are computed separately for corn and soybeans and for each model-feature specification.

In [25]:
# ============================================================
# C. Out-of-Sample Forecast Evaluation
# ============================================================

BENCHMARK = "Historical Mean"
METRICS_DIR = RESULTS / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Calculate forecasting performance metrics
# ------------------------------------------------------------

def calculate_metrics(group, benchmark_predictions):

    group = group.set_index("date").sort_index()

    # Align benchmark forecasts with the model's dates.
    benchmark = benchmark_predictions.reindex(group.index)

    actual = group["actual"].to_numpy()
    predicted = group["forecast"].to_numpy()
    benchmark_predicted = benchmark.to_numpy()

    if np.isnan(actual).any() or np.isnan(predicted).any():
        raise ValueError("Missing actual values or model forecasts.")

    if np.isnan(benchmark_predicted).any():
        raise ValueError("Missing benchmark forecasts.")

    errors = actual - predicted
    benchmark_errors = actual - benchmark_predicted

    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))

    benchmark_sse = np.sum(benchmark_errors ** 2)
    model_sse = np.sum(errors ** 2)

    if benchmark_sse == 0:
        oos_r2 = np.nan
    else:
        oos_r2 = 1 - model_sse / benchmark_sse

    directional_accuracy = np.mean(
        np.sign(actual) == np.sign(predicted)
    )

    return {
        "n_obs": len(group),
        "OOS_R2": oos_r2,
        "RMSE": rmse,
        "MAE": mae,
        "Directional_Accuracy": directional_accuracy,
    }


# ------------------------------------------------------------
# 2. Evaluate overall and annual performance
# ------------------------------------------------------------

def evaluate_all(predictions):

    summary_results = []
    annual_results = []

    predictions = predictions.reset_index()

    for commodity in sorted(predictions["commodity"].unique()):

        commodity_data = predictions.loc[
            predictions["commodity"] == commodity
        ].copy()

        benchmark_data = commodity_data.loc[
            commodity_data["model"] == BENCHMARK
        ].set_index("date").sort_index()

        if benchmark_data.index.has_duplicates:
            raise ValueError(
                f"Duplicate benchmark dates for {commodity}."
            )

        benchmark_predictions = benchmark_data["forecast"]

        # Overall out-of-sample performance.
        for (feature_set, model_name), group in (
            commodity_data.loc[
                commodity_data["model"] != BENCHMARK
            ].groupby(["feature_set", "model"])
        ):

            metrics = calculate_metrics(
                group,
                benchmark_predictions
            )

            summary_results.append({
                "commodity": commodity,
                "feature_set": feature_set,
                "model": model_name,
                **metrics,
            })

        # Historical-mean benchmark.
        benchmark_group = commodity_data.loc[
            commodity_data["model"] == BENCHMARK
        ]

        benchmark_metrics = calculate_metrics(
            benchmark_group,
            benchmark_predictions
        )

        summary_results.append({
            "commodity": commodity,
            "feature_set": "M0_Benchmark",
            "model": BENCHMARK,
            **benchmark_metrics,
        })

        # Annual performance.
        for year in sorted(commodity_data["date"].dt.year.unique()):

            if year < FIRST_TEST_YEAR:
                continue

            annual_data = commodity_data.loc[
                commodity_data["date"].dt.year == year
            ]

            annual_benchmark = benchmark_data.loc[
                benchmark_data.index.year == year,
                "forecast"
            ]

            for (feature_set, model_name), group in (
                annual_data.loc[
                    annual_data["model"] != BENCHMARK
                ].groupby(["feature_set", "model"])
            ):

                metrics = calculate_metrics(
                    group,
                    annual_benchmark
                )

                annual_results.append({
                    "commodity": commodity,
                    "year": year,
                    "feature_set": feature_set,
                    "model": model_name,
                    **metrics,
                })

            # Annual historical-mean benchmark.
            annual_benchmark_group = annual_data.loc[
                annual_data["model"] == BENCHMARK
            ]

            metrics = calculate_metrics(
                annual_benchmark_group,
                annual_benchmark
            )

            annual_results.append({
                "commodity": commodity,
                "year": year,
                "feature_set": "M0_Benchmark",
                "model": BENCHMARK,
                **metrics,
            })

    return (
        pd.DataFrame(summary_results),
        pd.DataFrame(annual_results),
    )


# ------------------------------------------------------------
# 3. Run the evaluation
# ------------------------------------------------------------

summary, annual = evaluate_all(forecasts)


# ------------------------------------------------------------
# 4. Save the evaluation results
# ------------------------------------------------------------

summary.to_csv(
    METRICS_DIR / "metrics_summary.csv",
    index=False
)

annual.to_csv(
    METRICS_DIR / "metrics_annual.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Display overall results
# ------------------------------------------------------------

for commodity in sorted(summary["commodity"].unique()):

    print(f"\n{'=' * 75}")
    print(f"OUT-OF-SAMPLE PERFORMANCE: {commodity.upper()}")
    print(f"{'=' * 75}")

    results = summary.loc[
        summary["commodity"] == commodity
    ].sort_values(
        ["feature_set", "OOS_R2"],
        ascending=[True, False]
    )

    display(
        results.style.format({
            "OOS_R2": "{:.4f}",
            "RMSE": "{:.6f}",
            "MAE": "{:.6f}",
            "Directional_Accuracy": "{:.2%}",
        })
    )


print("\nAnnual evaluation dataset:", annual.shape)
print("Overall evaluation dataset:", summary.shape)

print("\nSaved files:")
print(METRICS_DIR / "metrics_summary.csv")
print(METRICS_DIR / "metrics_annual.csv")



OUT-OF-SAMPLE PERFORMANCE: CORN


,commodity,feature_set,model,n_obs,OOS_R2,RMSE,MAE,Directional_Accuracy
9,corn,M0_Benchmark,Historical Mean,821,0.0000,0.034540,0.025315,47.87%
2,corn,M1_Financial,Ridge,821,-0.0056,0.034636,0.025375,51.52%
1,corn,M1_Financial,Random Forest,821,-0.0123,0.034752,0.025448,52.25%
0,corn,M1_Financial,HistGradientBoosting,821,-0.1385,0.036854,0.027623,48.36%
5,corn,M2_WSI,Ridge,821,-0.0042,0.034612,0.025405,52.62%
4,corn,M2_WSI,Random Forest,821,-0.0168,0.034828,0.025455,52.50%
3,corn,M2_WSI,HistGradientBoosting,821,-0.1394,0.036868,0.027648,47.99%
7,corn,M3_Components,Random Forest,821,-0.0066,0.034653,0.025433,51.40%
8,corn,M3_Components,Ridge,821,-0.0102,0.034716,0.025490,52.01%
6,corn,M3_Components,HistGradientBoosting,821,-0.1333,0.036769,0.027429,51.16%



OUT-OF-SAMPLE PERFORMANCE: SOYBEANS


,commodity,feature_set,model,n_obs,OOS_R2,RMSE,MAE,Directional_Accuracy
19,soybeans,M0_Benchmark,Historical Mean,834,0.0000,0.027510,0.021235,52.28%
12,soybeans,M1_Financial,Ridge,834,-0.0011,0.027525,0.021237,54.20%
11,soybeans,M1_Financial,Random Forest,834,-0.0057,0.027588,0.021450,50.84%
10,soybeans,M1_Financial,HistGradientBoosting,834,-0.0773,0.028553,0.022268,50.60%
15,soybeans,M2_WSI,Ridge,834,-0.0006,0.027518,0.021177,54.80%
14,soybeans,M2_WSI,Random Forest,834,-0.0040,0.027565,0.021394,52.64%
13,soybeans,M2_WSI,HistGradientBoosting,834,-0.1126,0.029017,0.022482,49.88%
18,soybeans,M3_Components,Ridge,834,-0.0015,0.027530,0.021234,56.24%
17,soybeans,M3_Components,Random Forest,834,-0.0051,0.027580,0.021401,51.68%
16,soybeans,M3_Components,HistGradientBoosting,834,-0.1290,0.029229,0.022658,48.44%



Annual evaluation dataset: (320, 9)
Overall evaluation dataset: (20, 8)

Saved files:
/Users/quentinfosalva/Documents/Python_Projects/weather-alpha-for-commodities/results/metrics/metrics_summary.csv
/Users/quentinfosalva/Documents/Python_Projects/weather-alpha-for-commodities/results/metrics/metrics_annual.csv



### Interpretation of Out-of-Sample Results

The out-of-sample results show that none of the machine learning models consistently outperforms the expanding historical-mean benchmark in terms of squared forecast errors. All models achieve slightly negative OOS $R^2$ values, indicating that their forecasts do not improve upon the benchmark over the evaluation period. This is particularly pronounced for HistGradientBoosting, which exhibits substantially higher RMSE and MAE for both commodities.

The inclusion of weather information produces modest improvements for Ridge regression. For corn, the WSI specification (M2) improves OOS $R^2$ from -0.56% to -0.42% relative to the financial-only specification (M1). For soybeans, it improves from -0.11% to -0.06%. These differences are small, however, and do not establish that weather information provides statistically significant predictive value.

Directional accuracy gives a somewhat different perspective. Several specifications achieve directional accuracy above 50%, with the highest value observed for the soybean Ridge model using individual weather components (M3), at 56.24%. However, higher directional accuracy does not necessarily translate into lower forecast errors, as illustrated by the negative OOS $R^2$ values.

Overall, the results provide limited preliminary evidence that weather variables may contain additional information about future commodity returns, particularly when incorporated into Ridge regression. Nevertheless, the improvements are economically and statistically uncertain. Formal forecast-comparison tests are therefore required before drawing conclusions about the incremental predictive value of weather information.



### F. Statistical Forecast Comparison Tests

To determine whether differences in out-of-sample predictive accuracy are statistically significant, paired forecast comparison tests are conducted.

**1. Diebold–Mariano test**

The Diebold–Mariano (DM) test compares the squared forecast errors of two competing models. The loss differential is defined as:

$$
d_t = e_{B,t}^{2} - e_{A,t}^{2}
$$

where $e_{A,t}$ and $e_{B,t}$ denote the forecast errors of models A and B, respectively. A positive mean differential indicates that model A has lower squared forecast errors than model B.

The null hypothesis is equal predictive accuracy, against a two-sided alternative:

$$
H_0: E[d_t] = 0
\qquad
H_1: E[d_t] \neq 0
$$

**2. Clark–West test**

For nested forecasting models, the Clark–West test adjusts the loss differential to account for the estimation noise introduced by additional predictors.

The adjusted loss differential is:

$$
d_t^{CW}
=
e_{R,t}^{2}
-
e_{A,t}^{2}
+
\left(\hat{y}_{R,t}-\hat{y}_{A,t}\right)^2
$$

where $R$ denotes the restricted financial-only model and $A$ the augmented model incorporating weather information.

The test uses a one-sided alternative:

$$
H_0: E[d_t^{CW}] \leq 0
\qquad
H_1: E[d_t^{CW}] > 0
$$

A positive mean adjusted differential favours the augmented model.

**3. HAC variance estimation**

Both tests use a Newey–West heteroskedasticity and autocorrelation consistent (HAC) estimator of the variance of the mean loss differential, with Bartlett weights and four lags in the baseline specification.

The variance estimator is:

$$
\widehat{\operatorname{Var}}(\bar d)
=
\frac{1}{n}
\left[
\hat{\gamma}_0
+
2\sum_{j=1}^{L}
\left(1-\frac{j}{L+1}\right)\hat{\gamma}_j
\right]
$$

where $L=4$ is the maximum number of lags and $\hat{\gamma}_j$ denotes the sample autocovariance at lag $j$.

**4. Forecast comparisons**

The following comparisons are performed separately for corn and soybeans:

- Each machine learning model against the historical-mean benchmark, using the DM test.
- Weather-augmented specifications (M2 and M3) against the financial-only specification (M1). The Clark–West test is used for Ridge, while the DM test is used for Random Forest and HistGradientBoosting.
- The WSI specification (M2) against the individual weather components specification (M3), using the DM test.

All comparisons use paired forecasts on common dates to ensure that the models are evaluated on identical observations.


In [31]:
# ============================================================
# F. Statistical Forecast Comparison Tests
# ============================================================

HAC_LAGS = 4

ML_MODELS = [
    "Ridge",
    "Random Forest",
    "HistGradientBoosting"
]

FEATURE_SETS_LIST = [
    "M1_Financial",
    "M2_WSI",
    "M3_Components"
]

BENCHMARK_MODEL = "Historical Mean"
BENCHMARK_FEATURE_SET = "Benchmark"

STAT_TEST_DIR = RESULTS / "statistical_tests"
STAT_TEST_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# HAC variance estimator
# ============================================================

def hac_variance_of_mean(values, max_lags=4):
    """
    Newey-West HAC estimator of the variance of the sample mean.
    Uses Bartlett weights.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    n = len(values)

    if n < 2:
        return np.nan

    centered = values - values.mean()

    gamma_0 = np.dot(centered, centered) / n
    long_run_variance = gamma_0

    max_lags = min(max_lags, n - 1)

    for lag in range(1, max_lags + 1):
        weight = 1 - lag / (max_lags + 1)

        gamma_lag = (
            np.dot(centered[lag:], centered[:-lag]) / n
        )

        long_run_variance += 2 * weight * gamma_lag

    return max(long_run_variance, 0) / n


# ============================================================
# Forecast comparison tests
# ============================================================

def test_statistic(loss_differential, alternative="two-sided"):
    """
    Test whether the mean loss differential is zero.
    """
    d = np.asarray(loss_differential, dtype=float)
    d = d[np.isfinite(d)]

    n = len(d)

    if n < 2:
        return np.nan, np.nan, n, np.nan

    mean_d = d.mean()
    variance_mean = hac_variance_of_mean(d, HAC_LAGS)

    if not np.isfinite(variance_mean) or variance_mean <= 0:
        return mean_d, np.nan, n, np.nan

    standard_error = np.sqrt(variance_mean)
    statistic = mean_d / standard_error

    if alternative == "two-sided":
        p_value = 2 * stats.t.sf(abs(statistic), df=n - 1)

    elif alternative == "greater":
        p_value = stats.t.sf(statistic, df=n - 1)

    else:
        raise ValueError("Unknown alternative.")

    return mean_d, statistic, n, p_value


def diebold_mariano(actual, forecast_a, forecast_b):
    """
    DM test using squared-error loss.

    Positive mean differential means forecast A has lower
    squared forecast error than forecast B.
    """
    error_a = actual - forecast_a
    error_b = actual - forecast_b

    differential = error_b**2 - error_a**2

    return test_statistic(
        differential,
        alternative="two-sided"
    )


def clark_west(actual, forecast_restricted, forecast_augmented):
    """
    Clark-West test for nested forecasting models.

    Restricted model: financial-only model.
    Augmented model: financial + weather model.
    """
    error_restricted = actual - forecast_restricted
    error_augmented = actual - forecast_augmented

    adjustment = (
        forecast_restricted - forecast_augmented
    ) ** 2

    differential = (
        error_restricted**2
        - error_augmented**2
        + adjustment
    )

    return test_statistic(
        differential,
        alternative="greater"
    )


# ============================================================
# Prepare paired forecasts
# ============================================================

def get_paired_forecasts(
    df, commodity, model_a, feature_a,
    model_b, feature_b
):

    df = df.reset_index()

    a = df[
        (df["commodity"] == commodity)
        & (df["model"] == model_a)
        & (df["feature_set"] == feature_a)
    ][["date", "actual", "forecast"]].copy()

    b = df[
        (df["commodity"] == commodity)
        & (df["model"] == model_b)
        & (df["feature_set"] == feature_b)
    ][["date", "actual", "forecast"]].copy()

    a = a.rename(columns={
        "actual": "actual_a",
        "forecast": "forecast_a",
    })

    b = b.rename(columns={
        "actual": "actual_b",
        "forecast": "forecast_b",
    })

    paired = a.merge(b, on="date", how="inner")

    if not np.allclose(
        paired["actual_a"],
        paired["actual_b"],
        equal_nan=True
    ):
        raise ValueError(
            f"Actual returns do not match: {commodity}, "
            f"{model_a}/{feature_a}, {model_b}/{feature_b}"
        )

    paired = paired.dropna()
    paired["actual"] = paired["actual_a"]

    return paired[
        ["date", "actual", "forecast_a", "forecast_b"]
    ].sort_values("date")


# ============================================================
# Run all comparisons
# ============================================================

def run_tests(df):
    results = []

    def add_result(
        commodity, comparison, test_name,
        mean_diff, statistic, n, p_value,
        alternative, interpretation
    ):
        results.append({
            "commodity": commodity,
            "comparison": comparison,
            "test": test_name,
            "n_obs": n,
            "mean_adjusted_loss_difference": mean_diff,
            "test_statistic": statistic,
            "p_value": p_value,
            "alternative": alternative,
            "interpretation": interpretation,
        })

    for commodity in COMMODITIES:

        # ----------------------------------------------------
        # A. ML models versus historical mean benchmark
        # ----------------------------------------------------

        for feature_set in FEATURE_SETS_LIST:

            for model in ML_MODELS:

                paired = get_paired_forecasts(
                    df,
                    commodity,
                    model,
                    feature_set,
                    BENCHMARK_MODEL,
                    BENCHMARK_FEATURE_SET
                )

                mean_d, statistic, n, p_value = diebold_mariano(
                    paired["actual"].values,
                    paired["forecast_a"].values,
                    paired["forecast_b"].values
                )

                interpretation = (
                    "ML model has lower squared error"
                    if mean_d > 0
                    else "Historical mean has lower squared error"
                )

                add_result(
                    commodity,
                    f"{feature_set}_{model}_vs_HistoricalMean",
                    "Diebold-Mariano",
                    mean_d,
                    statistic,
                    n,
                    p_value,
                    "two-sided",
                    interpretation
                )

        # ----------------------------------------------------
        # B. Weather versus financial-only models
        # ----------------------------------------------------

        for model in ML_MODELS:

            for weather_set in ["M2_WSI", "M3_Components"]:

                paired = get_paired_forecasts(
                    df,
                    commodity,
                    model,
                    "M1_Financial",
                    model,
                    weather_set
                )

                if model == "Ridge":

                    mean_d, statistic, n, p_value = clark_west(
                        paired["actual"].values,
                        paired["forecast_a"].values,
                        paired["forecast_b"].values
                    )

                    test_name = "Clark-West"
                    alternative = "one-sided: weather improves"

                else:

                    mean_d, statistic, n, p_value = diebold_mariano(
                        paired["actual"].values,
                        paired["forecast_b"].values,
                        paired["forecast_a"].values
                    )

                    test_name = "Diebold-Mariano"
                    alternative = "two-sided"

                interpretation = (
                    "Weather model improves forecast accuracy"
                    if mean_d > 0
                    else "Financial-only model has lower adjusted loss"
                )

                add_result(
                    commodity,
                    f"{weather_set}_{model}_vs_M1_Financial",
                    test_name,
                    mean_d,
                    statistic,
                    n,
                    p_value,
                    alternative,
                    interpretation
                )

        # ----------------------------------------------------
        # C. DM: WSI versus individual components
        # ----------------------------------------------------

        for model in ML_MODELS:

            paired = get_paired_forecasts(
                df,
                commodity,
                model,
                "M2_WSI",
                model,
                "M3_Components"
            )

            mean_d, statistic, n, p_value = diebold_mariano(
                paired["actual"].values,
                paired["forecast_a"].values,
                paired["forecast_b"].values
            )

            interpretation = (
                "WSI has lower squared error"
                if mean_d > 0
                else "Individual components have lower squared error"
            )

            add_result(
                commodity,
                f"M2_WSI_{model}_vs_M3_Components",
                "Diebold-Mariano",
                mean_d,
                statistic,
                n,
                p_value,
                "two-sided",
                interpretation
            )

    results_df = pd.DataFrame(results)

    results_df["significant_10pct"] = (
        results_df["p_value"] < 0.10
    )

    results_df["significant_5pct"] = (
        results_df["p_value"] < 0.05
    )

    results_df["significant_1pct"] = (
        results_df["p_value"] < 0.01
    )

    output_path = STAT_TEST_DIR / "forecast_comparison_tests.csv"
    results_df.to_csv(output_path, index=False)

    print("\nStatistical tests completed.")
    print(f"Results saved to: {output_path}")

    print("\nResults by commodity and test:")
    print(
        results_df.groupby(
            ["commodity", "test"]
        ).size().to_string()
    )

    print("\nSignificant results at 5%:")

    significant = results_df[
        results_df["significant_5pct"]
    ]

    if significant.empty:
        print("No statistically significant results at the 5% level.")
    else:
        print(
            significant[
                [
                    "commodity",
                    "comparison",
                    "test",
                    "mean_adjusted_loss_difference",
                    "test_statistic",
                    "p_value",
                ]
            ].to_string(index=False)
        )

    return results_df


# ============================================================
# Main
# ============================================================

statistical_results = run_tests(forecasts)



Statistical tests completed.
Results saved to: /Users/quentinfosalva/Documents/Python_Projects/weather-alpha-for-commodities/results/statistical_tests/forecast_comparison_tests.csv

Results by commodity and test:
commodity  test           
corn       Clark-West          2
           Diebold-Mariano    16
soybeans   Clark-West          2
           Diebold-Mariano    16

Significant results at 5%:
commodity                                           comparison            test  mean_adjusted_loss_difference  test_statistic  p_value
     corn  M1_Financial_HistGradientBoosting_vs_HistoricalMean Diebold-Mariano                      -0.000165       -3.826111 0.000140
     corn        M2_WSI_HistGradientBoosting_vs_HistoricalMean Diebold-Mariano                      -0.000166       -4.398004 0.000012
     corn M3_Components_HistGradientBoosting_vs_HistoricalMean Diebold-Mariano                      -0.000159       -3.649877 0.000279
 soybeans  M1_Financial_HistGradientBoosting_vs_HistoricalM


### Interpretation of Statistical Forecast Comparison Tests

The statistical tests provide a more nuanced picture than the preliminary performance metrics.

**1. Machine learning models versus the historical mean**

The Diebold–Mariano tests identify statistically significant differences for all six HistGradientBoosting specifications. However, the negative mean adjusted loss differentials indicate that HistGradientBoosting has a higher squared forecast error than the historical mean. This confirms that its negative out-of-sample R² is not merely a numerical difference: its forecasting performance is significantly worse than the benchmark for both commodities, regardless of the feature set.

No significant evidence is found that Ridge or Random Forest outperform the historical mean.

**2. Weather information versus financial information**

The Clark–West and Diebold–Mariano tests compare models incorporating weather information with their financial-only counterparts.

For corn, neither the WSI nor the individual weather components produce a statistically significant improvement over the financial-only specification.

For soybeans, the Clark–West test identifies a significant improvement at the 5% level when Ridge incorporates individual weather components (M3) rather than financial features alone (M1), with a p-value of 0.0208. This provides evidence that weather components contain additional predictive information for soybean returns within this specification.

However, this result should be interpreted cautiously. It is the only significant result among the weather-versus-financial comparisons, and the Clark–West test is applied to regularised Ridge models.

**3. Composite WSI versus individual weather components**

The results do not identify a statistically significant difference between the composite WSI (M2) and the individual weather components (M3). Consequently, there is no evidence that either representation systematically provides superior forecasting performance.

**Overall conclusion**

The statistical tests suggest that weather information may provide some incremental predictive value for soybean returns, particularly when individual weather components are incorporated into a Ridge model. However, this evidence is limited and does not establish a systematic improvement across commodities and model specifications.

Importantly, the significant deterioration in HistGradientBoosting performance highlights that greater model complexity does not necessarily translate into better out-of-sample forecasts. The historical mean remains a difficult benchmark to beat.


## V. Robustness Analysis

### Sensitivity to HAC Lag Selection

The statistical tests presented in Section IV.F rely on a heteroskedasticity- and autocorrelation-consistent (HAC) estimator to calculate the variance of the mean loss differential. The choice of the maximum lag used in this estimator can influence the standard errors, test statistics and p-values.

To assess whether our statistical conclusions depend on this choice, we repeat the forecast-comparison tests using three different HAC lag specifications: 0, 4 and 8.

The zero-lag specification accounts for heteroskedasticity without correcting for serial correlation, while the four- and eight-lag specifications additionally account for autocorrelation over different time horizons. The baseline specification uses four lags.

We compare the resulting test statistics and significance levels across these specifications. The objective is to verify whether the conclusions regarding the relative performance of the forecasting models and the contribution of weather information remain stable under alternative HAC assumptions.

In [32]:
# Robustness analysis: sensitivity to HAC lag selection

HAC_LAGS_BASELINE = 4
HAC_LAG_SPECS = [0, 4, 8]

hac_results = []

try:
    for lag in HAC_LAG_SPECS:
        HAC_LAGS = lag

        lag_results = run_tests(forecasts)
        lag_results["hac_lags"] = lag

        hac_results.append(lag_results)

finally:
    # Restore the baseline HAC specification
    HAC_LAGS = HAC_LAGS_BASELINE

# Combine results across all HAC specifications
hac_robustness = pd.concat(hac_results, ignore_index=True)

hac_robustness = hac_robustness.sort_values(
    ["commodity", "comparison", "hac_lags"]
).reset_index(drop=True)

# Save robustness results
output_path = RESULTS / "statistical_tests" / "hac_robustness.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

hac_robustness.to_csv(output_path, index=False)

print(f"Robustness results saved to: {output_path}")
print(f"Shape: {hac_robustness.shape}")

display(hac_robustness)



Statistical tests completed.
Results saved to: /Users/quentinfosalva/Documents/Python_Projects/weather-alpha-for-commodities/results/statistical_tests/forecast_comparison_tests.csv

Results by commodity and test:
commodity  test           
corn       Clark-West          2
           Diebold-Mariano    16
soybeans   Clark-West          2
           Diebold-Mariano    16

Significant results at 5%:
commodity                                           comparison            test  mean_adjusted_loss_difference  test_statistic  p_value
     corn  M1_Financial_HistGradientBoosting_vs_HistoricalMean Diebold-Mariano                      -0.000165       -4.083044 0.000049
     corn        M2_WSI_HistGradientBoosting_vs_HistoricalMean Diebold-Mariano                      -0.000166       -4.561253 0.000006
     corn M3_Components_HistGradientBoosting_vs_HistoricalMean Diebold-Mariano                      -0.000159       -4.040158 0.000058
 soybeans  M1_Financial_HistGradientBoosting_vs_HistoricalM

,commodity,comparison,test,n_obs,mean_adjusted_loss_difference,test_statistic,p_value,alternative,interpretation,significant_10pct,significant_5pct,significant_1pct,hac_lags
0,corn,M1_Financial_HistGradientBoosting_vs_Historica...,Diebold-Mariano,821,-0.000165,-4.083044,0.000049,two-sided,Historical mean has lower squared error,True,True,True,0
1,corn,M1_Financial_HistGradientBoosting_vs_Historica...,Diebold-Mariano,821,-0.000165,-3.826111,0.000140,two-sided,Historical mean has lower squared error,True,True,True,4
2,corn,M1_Financial_HistGradientBoosting_vs_Historica...,Diebold-Mariano,821,-0.000165,-3.958159,0.000082,two-sided,Historical mean has lower squared error,True,True,True,8
3,corn,M1_Financial_Random Forest_vs_HistoricalMean,Diebold-Mariano,821,-0.000015,-1.008704,0.313414,two-sided,Historical mean has lower squared error,False,False,False,0
4,corn,M1_Financial_Random Forest_vs_HistoricalMean,Diebold-Mariano,821,-0.000015,-0.890229,0.373604,two-sided,Historical mean has lower squared error,False,False,False,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,soybeans,M3_Components_Ridge_vs_HistoricalMean,Diebold-Mariano,834,-0.000001,-0.122619,0.902439,two-sided,Historical mean has lower squared error,False,False,False,4
104,soybeans,M3_Components_Ridge_vs_HistoricalMean,Diebold-Mariano,834,-0.000001,-0.117807,0.906249,two-sided,Historical mean has lower squared error,False,False,False,8
105,soybeans,M3_Components_Ridge_vs_M1_Financial,Clark-West,834,0.000016,2.181613,0.014708,one-sided: weather improves,Weather model improves forecast accuracy,True,True,False,0
106,soybeans,M3_Components_Ridge_vs_M1_Financial,Clark-West,834,0.000016,2.041353,0.020765,one-sided: weather improves,Weather model improves forecast accuracy,True,True,False,4


### Results and Interpretation

The results remain stable across the three HAC specifications (0, 4 and 8 lags).

HistGradientBoosting significantly underperforms the historical-mean benchmark for both commodities and all feature sets, with p-values below 1% in every specification.

For soybeans, the Ridge model incorporating individual weather components (M3) significantly outperforms its financial-only counterpart (M1). This result remains significant at 5% across all specifications, although not at 1%.

Overall, the conclusions are robust to the choice of HAC lag length. However, evidence of additional predictive value from weather information remains limited.

## VI. Conclusion

This project investigated whether incorporating weather information improves the out-of-sample prediction of agricultural commodity returns relative to models relying exclusively on financial information. The analysis focused on corn and soybean futures, combining historical market data with production-weighted weather indicators and a composite Weather Stress Index (WSI).

The empirical results provide **limited evidence that weather information improves predictive performance**. Across both commodities, machine learning models generally failed to outperform the expanding historical-mean benchmark in terms of out-of-sample squared prediction errors. Ridge regression delivered the most competitive results, while HistGradientBoosting consistently exhibited weaker forecasting performance.

Comparing the different feature sets reveals modest improvements from incorporating weather information. For Ridge regression, adding the WSI slightly improved out-of-sample $R^2$) for both commodities relative to the financial-only specification. The model using individual weather components also achieved the highest directional accuracy for soybeans, correctly predicting the sign of weekly returns in 56.24% of observations. However, these improvements should be interpreted cautiously: better directional accuracy does not necessarily translate into lower prediction errors.

The statistical analysis provides further qualification. The Diebold–Mariano and Clark–West tests indicate that most improvements associated with weather information are not statistically significant. One exception is the soybean Ridge specification incorporating individual weather components, which outperforms its financial-only counterpart at the 5% significance level. This result remains significant across the alternative HAC lag specifications, although it should be treated as suggestive rather than conclusive given the number of comparisons performed.

Several limitations should be considered. First, the analysis focuses on only two agricultural commodities and a one-week forecasting horizon. Second, the construction of the WSI relies on an adapted methodology and historical weather observations rather than real-time weather forecasts. Consequently, the results do not establish whether forecasted weather information could provide additional predictive value. Finally, the limited performance of the models suggests that weekly commodity returns remain difficult to predict using the selected financial and meteorological variables.

Future research could extend the analysis to additional agricultural commodities, investigate longer forecasting horizons, and incorporate weather forecasts alongside historical observations. More sophisticated measures of weather uncertainty and crop-specific production conditions could also provide a richer representation of agricultural supply shocks.

**Overall, the findings suggest that weather information contains some potentially useful information for forecasting agricultural commodity returns, particularly for soybeans, but its incremental predictive value remains modest and is not consistently supported by statistical evidence.**